In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw_pushshift.csv")
print(df.shape)
df.head(10)

(29195910, 7)


,id,subreddit,dialect,type,text,score,created_utc
0,c058wdb,argentina,rioplatense,comment,Bienvenidos al reddit de Argentina! Aunque fue...,2,1.220229e+09
1,c05dkl0,argentina,rioplatense,comment,Soy parte de esta gran comunidad :P,2,1.221071e+09
2,c05knd3,argentina,rioplatense,comment,"interesante sitio, bien simple y al punto, yo ...",2,1.222323e+09
3,c05pqr6,argentina,rioplatense,comment,Estaba mirando el perfil y acabo de notar que ...,2,1.223186e+09
4,c0697cz,argentina,rioplatense,comment,El plan de desactivar el subsistema de capital...,2,1.226152e+09
5,c06bsss,argentina,rioplatense,comment,Muy buena idea. A ver cuantos redditeros argen...,1,1.226525e+09
6,c06cudo,argentina,rioplatense,comment,"no sos de acá? \n\nes como tu...tu y eres, per...",1,1.226675e+09
7,c06d2mi,argentina,rioplatense,comment,"vos no se conjuga, no es un verbo\nse más espe...",1,1.226698e+09
8,c06evnx,argentina,rioplatense,comment,Aquí está todo lo que necesites saber al respe...,1,1.227009e+09
9,c06l33i,argentina,rioplatense,comment,quiero este programa!,1,1.228005e+09


In [7]:
df.groupby(["dialect", "type"]).size().unstack()

type,comment,post
dialect,,
cdmx,9186454,528765
madrid,419309,93044
rioplatense,18329673,638665


In [8]:
for dialect in ["rioplatense", "cdmx", "madrid"]:
    print(f"\n--- {dialect} ---")
    sample = df[df.dialect == dialect]["text"].sample(5)
    for text in sample:
        print(text)
        print()
        


--- rioplatense ---
Se parece al fisura del barrio

Un auto no pero una casa no es algo sencillo en ningún lado casi…

mercado libre saca ganancias de este acto solidario?

No son dos barrios. Es un solo barrio. Palermo. Después hay zonas ahi dentro, pero es un solo barrio. 

El de Paraná es este: 6FQ7+77 Paraná, Entre Ríos


Se tardó demasiado en terminar

Me habían dicho que las cuotas son de alrededor de 40k pesos

Tuvieron un problema con las cloacas pero parece que lo solucionaron


--- cdmx ---
Muy cierto.

A Oliver North y a cierto ex-Gobernador de Arkansas no les gusta esto :D

Magdalenas

Los estándares de belleza tienen poco que ver con el poderío que el patriarcado lleva a cabo. Estos cambian con los siglos y las tendencias, pues fácilmente alguien puede decir que una persona se ve fea o bonita en base a lo que aprende de la sociedad que la rodea en su momento. Nada tiene que ver el machismo en mi opinión pues los prejuicios basados en imagen los da cualquiera. Yo aconsejar

In [5]:
df["text_length"] = df["text"].str.len()
df.groupby("dialect")["text_length"].describe()


,count,mean,std,min,25%,50%,75%,max
dialect,,,,,,,,
cdmx,9715191.0,183.021223,312.844342,1.0,40.0,89.0,202.0,39409.0
madrid,512350.0,195.352091,339.959704,1.0,42.0,92.0,222.0,29251.0
rioplatense,18968298.0,189.044581,491.507751,1.0,39.0,83.0,179.0,39465.0


DROPPING ALL ENTRIES UNDER LENGTH 20

In [16]:
df_1 = df[df["text"].str.len() >= 20]
print(f"After length filter: {df.groupby('dialect').size()}")

After length filter: dialect
cdmx            9715219
madrid           512353
rioplatense    18968338
dtype: int64


PICKING A RANDOM SAMPLE FROM EACH DIALECT DUE TO DIFFERENCE IN SIZES (AND PROCESSING CONSTRAINTS)

In [17]:
min_size = df_1.groupby("dialect").size().min()
buffer_size = min_size * 3

sampled_parts = []
for dialect, group in df_1.groupby("dialect"):
    n = min(len(group), buffer_size)
    sampled_parts.append(group.sample(n=n, random_state=42))

df_2 = pd.concat(sampled_parts, ignore_index=True)
print(f"After buffered sample:\n{df_2.groupby('dialect').size()}")

After buffered sample:
dialect
cdmx           1383252
madrid          461084
rioplatense    1383252
dtype: int64


REMOVING ENTRIES IN OTHER LANGUAGES

In [18]:
from langdetect import detect, LangDetectException
from tqdm import tqdm

tqdm.pandas(desc="Detecting language")

def is_spanish(text):
    try:
        return detect(text) == "es"
    except LangDetectException:
        return False

print("Running language detection, this will take a while...")
df_2["is_spanish"] = df_2["text"].progress_apply(is_spanish)
df_3 = df_2[df_2["is_spanish"]].drop(columns="is_spanish")
print(f"After language filter:\n{df_3.groupby('dialect').size()}")

Running language detection, this will take a while...


Detecting language: 100%|██████████| 3227588/3227588 [5:14:52<00:00, 170.84it/s]  


After language filter:
dialect
cdmx           1230100
madrid          310820
rioplatense    1236316
dtype: int64


REMOVING COPY PASTES/DUPES *UNTESTED

In [31]:
# Exact duplicates
df_4 = df_3.drop_duplicates(subset=['text'], keep='first')

print(f"Final dataset size: {len(df_4)}")

Final dataset size: 2748382


In [26]:
print(f"After language filter:\n{df_4.groupby('dialect').size()}")

After language filter:
dialect
cdmx           1214181
madrid          306545
rioplatense    1227656
dtype: int64


In [28]:
df_4["text_length"] = df_4["text"].str.len()
df_4.groupby("dialect")["text_length"].describe()

,count,mean,std,min,25%,50%,75%,max
dialect,,,,,,,,
cdmx,1214181.0,210.000958,328.637110,20.0,59.0,112.0,233.0,39338.0
madrid,306545.0,215.658915,363.059973,20.0,58.0,106.0,235.0,29251.0
rioplatense,1227656.0,221.928627,529.208146,20.0,57.0,106.0,212.0,31015.0


In [ ]:
for dialect in ["rioplatense", "cdmx", "madrid"]:
    print(f"\n--- {dialect} ---")
    sample = df_4[df_4.dialect == dialect]["text"].sample(5)
    for text in sample:
        print(text)
        print()


--- rioplatense ---
Hola chicos/as/es.Como andan?Todo bien?Yo por mi parte no.Les cuento ,hace un tiempo que estoy andando con una chica,es increíble le gusta lo mismo que a mí,es sensible ,me banca ,es única y la amo con toda mi alma, lamentablemente ella tiene muchos quilombos con su familia ,más que nada del lado de su mamá.Su mamá antes tenía quilombos con las drogas y tenía mala junta,ahora se rescató y ya está organizando su vida ,mi novia le prestó una plata para ayudarla y así y ella se la iba a devolver,pues la casa no tiene traba ni nada (para que vean el quilombo en el que estaba) y siempre le entra a robar un primo (de mi novia) ,pues adivinen,le robaron la plata que mi novia le prestó.Para colmo no es la primera vez que pasa,ya le han denunciado pero es menor y no pueden hacer nada.El asunto es: NESECITO AYUDA!!NO PUEDO CAGARLO A PIÑAS, NESECITO AYUDA DE USTEDES LEGAL O LO QUE SEA , PORFAVOR APORTENME.Gracias por ver.Espero que tengan un lindo dia

Es el arte de vender ba

In [30]:
print(df_4.columns)

Index(['id', 'subreddit', 'dialect', 'type', 'text', 'score', 'created_utc',
       'text_length'],
      dtype='str')


In [32]:
df_4.to_csv('cleaned_pushshift.csv', index=False)